# 05. Inference

This notebook is the standalone inference half of `02. mobilenet_model.ipynb`.

It starts from the inference section and keeps the later calibration, uncertainty,
real-world checks, targeted fine-tuning, productionization, segmentation, ROI crop,
and multi-crop sections.

The bootstrap cells below make it runnable without executing the development/training
part of notebook 02 first.

In [ ]:
import os
import io
import json
import copy
import shutil
from pathlib import Path
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset, Dataset
from torchvision import models, transforms
from torchvision.datasets import ImageFolder

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
ROOT = Path('..').resolve()
TRAIN_DIR = str(ROOT / 'data' / 'train')
VAL_DIR = str(ROOT / 'data' / 'val')
HARD_TRAIN_DIR = str(ROOT / 'data' / 'hard' / 'train')
HARD_VAL_DIR = str(ROOT / 'data' / 'hard' / 'val')
MODELS_DIR = ROOT / 'models'
OUTPUTS_DIR = ROOT / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

MOBILENET_MODEL_SAVE_PATH = str(MODELS_DIR / 'MobileNet_model.pth')
MOBILENET_FINETUNED_SAVE_PATH = str(MODELS_DIR / 'MobileNet_Finetuned_added_data.pth')

IMAGE_SIZE = 224
DATASET_MEAN = [0.502275, 0.528588, 0.474627]
DATASET_STD = [0.251866, 0.251658, 0.335502]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = DEVICE

print('ROOT:', ROOT)
print('DEVICE:', DEVICE)
print('TRAIN_DIR exists:', os.path.exists(TRAIN_DIR))
print('VAL_DIR exists  :', os.path.exists(VAL_DIR))

In [ ]:
def get_val_transform():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=DATASET_MEAN, std=DATASET_STD),
    ])


def get_light_transform():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=DATASET_MEAN, std=DATASET_STD),
    ])


def get_train_transform():
    return transforms.Compose([
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.4, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(90),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=DATASET_MEAN, std=DATASET_STD),
        transforms.RandomErasing(p=0.5, scale=(0.02, 0.25), value='random'),
    ])


def get_mobilenet_model(num_classes, version='v2', pretrained=True, dropout=0.2):
    if version != 'v2':
        raise ValueError('Standalone inference notebook currently supports MobileNetV2 only')
    weights = None
    model = models.mobilenet_v2(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return model


class RemappedImageFolder(Dataset):
    def __init__(self, imagefolder, expected_class_to_idx, filtered_samples=None):
        self.imagefolder = imagefolder
        self.transform = imagefolder.transform
        self.loader = imagefolder.loader
        self.classes = list(expected_class_to_idx.keys())
        self.class_to_idx = dict(expected_class_to_idx)
        source_samples = filtered_samples if filtered_samples is not None else imagefolder.samples
        self.samples = []
        for path, label in source_samples:
            class_name = imagefolder.classes[label]
            if class_name not in expected_class_to_idx:
                raise ValueError(f'Unknown class {class_name}')
            self.samples.append((path, expected_class_to_idx[class_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = self.loader(path)
        if self.transform is not None:
            img = self.transform(img)
        return img, label

In [ ]:
def resolve_checkpoint_path():
    candidates = [
        Path(MOBILENET_FINETUNED_SAVE_PATH),
        MODELS_DIR / 'MobileNet_targeted_hard_finetune.pth',
        MODELS_DIR / 'MobileNet_Finetuned_added_data.pth',
        MODELS_DIR / 'MobileNet_production.pth',
        MODELS_DIR / 'MobileNet_model.pth',
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f'No checkpoint found in {candidates}')


def load_checkpoint_classes(ckpt):
    if isinstance(ckpt, dict) and 'idx_to_class' in ckpt:
        return list(ckpt['idx_to_class'])
    if isinstance(ckpt, dict) and 'class_to_idx' in ckpt:
        inv = {v: k for k, v in ckpt['class_to_idx'].items()}
        return [inv[i] for i in range(len(inv))]
    meta = MODELS_DIR / 'model_metadata.json'
    if meta.exists():
        with meta.open('r', encoding='utf-8') as f:
            return list(json.load(f).get('classes', []))
    raise RuntimeError('Could not resolve class mapping from checkpoint or metadata')


checkpoint_path = resolve_checkpoint_path()
ckpt = torch.load(checkpoint_path, map_location=device)
classes = load_checkpoint_classes(ckpt)
num_classes = len(classes)
idx_to_class_runtime = list(classes)

model = get_mobilenet_model(num_classes, version='v2', dropout=0.2).to(device)
state_dict = ckpt['model_state'] if isinstance(ckpt, dict) and 'model_state' in ckpt else ckpt
model.load_state_dict(state_dict)
model.eval()

base_train = ImageFolder(root=TRAIN_DIR, transform=get_light_transform())
val_dataset = ImageFolder(root=VAL_DIR, transform=get_val_transform())
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

print('Loaded checkpoint:', checkpoint_path)
print('Classes:', classes)
print('Validation images:', len(val_dataset))

# Inference

## 16. Inference Pipeline

In [ ]:
from torchvision.datasets import ImageFolder
import torch.nn.functional as F

In [ ]:
model.eval()
confidences = []

n_samples = min(100, len(val_dataset))

with torch.no_grad():
    for i in range(n_samples):
        img_tensor, label = val_dataset[i]
        logits = model(img_tensor.unsqueeze(0).to(device))
        probs = F.softmax(logits, dim=1)
        conf, pred = torch.max(probs, dim=1)
        
        confidences.append(conf.item())

print(f"Samples checked: {n_samples}")
print(f"Average confidence: {np.mean(confidences):.2%}")
print(f"Min confidence: {np.min(confidences):.2%}")
print(f"Max confidence: {np.max(confidences):.2%}")


Inference on Validation Photo

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch.nn as nn


def _find_last_conv_layer(model):
    last_conv = None
    for _, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_conv = module
    if last_conv is None:
        raise RuntimeError('No Conv2d layer found for Grad-CAM target.')
    return last_conv


def show_gradcam(model, image_path, device, class_names, target_class_idx=None, alpha=0.45, title=None):
    model.eval()

    activations = None

    def forward_hook(_, __, output):
        nonlocal activations
        activations = output

    target_layer = _find_last_conv_layer(model)
    f_handle = target_layer.register_forward_hook(forward_hook)

    try:
        img = Image.open(image_path).convert('RGB')
        x = get_val_transform()(img).unsqueeze(0).to(device)
        x.requires_grad_(True)

        with torch.enable_grad():
            logits = model(x)
            probs = F.softmax(logits, dim=1)

            if target_class_idx is None:
                target_class_idx = int(torch.argmax(probs, dim=1).item())

            if activations is None:
                raise RuntimeError('Grad-CAM forward hook did not capture activations.')

            score = logits[0, target_class_idx]
            grads = torch.autograd.grad(score, activations, retain_graph=False, allow_unused=True)[0]
            if grads is None:
                raise RuntimeError('Grad-CAM could not compute gradients for target layer.')

            weights = grads.mean(dim=(2, 3), keepdim=True)
            cam = (weights * activations).sum(dim=1, keepdim=True)
            cam = torch.relu(cam)

            # Upsample low-res CAM to image resolution for smooth heatmap
            cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)

        cam = cam[0, 0].detach().cpu().numpy()
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()

        img_np = np.array(img.resize((224, 224))).astype(np.float32) / 255.0

        plt.figure(figsize=(6, 6))
        plt.imshow(img_np)
        plt.imshow(cam, cmap='jet', alpha=alpha, interpolation='bilinear')

        pred_idx = int(torch.argmax(probs, dim=1).item())
        pred_name = class_names[pred_idx]
        conf = float(probs[0, pred_idx].item())

        if title is None:
            title = f'Grad-CAM | Pred: {pred_name} ({conf:.1%})'
        plt.title(title)
        plt.axis('off')
        plt.show()

    finally:
        f_handle.remove()

print('Grad-CAM helper ready (smooth heatmap)')


In [ ]:
from PIL import Image

val_transform = get_val_transform()
image_path = r"C:\Users\loaim\OneDrive\Desktop\IMG_20211108_120942 (Custom).jpg"
img = Image.open(image_path).convert("RGB")
img_tensor = val_transform(img)

with torch.no_grad():
    model.eval()
    logits = model(img_tensor.unsqueeze(0).to(device))
    probs = F.softmax(logits, dim=1)
    conf, pred = torch.max(probs, dim=1)

print(f"Prediction: {classes[pred.item()]}")
print(f"Confidence: {conf.item():.2%}")

show_gradcam(
    model=model,
    image_path=image_path,
    device=device,
    class_names=classes,
    target_class_idx=int(pred.item()),
    title=f"Grad-CAM: {classes[pred.item()]} ({conf.item():.1%})",
)


### Validation on Real-world Image

In [ ]:
from PIL import Image

# Load model
ckpt = torch.load(MOBILENET_FINETUNED_SAVE_PATH, map_location=device)
model = get_mobilenet_model(len(classes), version='v2', dropout=0.2).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()

# Predict
image_path = r"C:\Users\loaim\OneDrive\Desktop\Untitled.jpg"
img = Image.open(image_path).convert("RGB")
img_tensor = get_val_transform()(img)

with torch.no_grad():
    logits = model(img_tensor.unsqueeze(0).to(device))
    probs = F.softmax(logits, dim=1)
    conf, pred = torch.max(probs, dim=1)

print(f"{classes[pred.item()]}: {conf.item():.2%}")

show_gradcam(
    model=model,
    image_path=image_path,
    device=device,
    class_names=classes,
    target_class_idx=int(pred.item()),
    title=f"Grad-CAM: {classes[pred.item()]} ({conf.item():.1%})",
)


In [ ]:
# Predict
image_path = r"C:\Users\loaim\OneDrive\Desktop\furtBovAv5YwEs53CCKHeC.jpg"
img = Image.open(image_path).convert("RGB")
img_tensor = get_val_transform()(img)

with torch.no_grad():
    logits = model(img_tensor.unsqueeze(0).to(device))
    probs = F.softmax(logits, dim=1)
    conf, pred = torch.max(probs, dim=1)

print(f"{classes[pred.item()]}: {conf.item():.2%}")

show_gradcam(
    model=model,
    image_path=image_path,
    device=device,
    class_names=classes,
    target_class_idx=int(pred.item()),
    title=f"Grad-CAM: {classes[pred.item()]} ({conf.item():.1%})",
)


In [ ]:
image_path = r"C:\Users\loaim\OneDrive\Desktop\stigminaleafspots.jpg"
img = Image.open(image_path).convert("RGB")
img_tensor = get_val_transform()(img)

with torch.no_grad():
    logits = model(img_tensor.unsqueeze(0).to(device))
    probs = F.softmax(logits, dim=1)
    conf, pred = torch.max(probs, dim=1)

print(f"{classes[pred.item()]}: {conf.item():.2%}")

show_gradcam(
    model=model,
    image_path=image_path,
    device=device,
    class_names=classes,
    target_class_idx=int(pred.item()),
    title=f"Grad-CAM: {classes[pred.item()]} ({conf.item():.1%})",
)


In [ ]:
image_path = r"C:\Users\loaim\OneDrive\Desktop\stigminaleafspots.jpg"
img = Image.open(image_path).convert("RGB")
img_tensor = get_val_transform()(img)

with torch.no_grad():
    logits = model(img_tensor.unsqueeze(0).to(device))
    probs = F.softmax(logits, dim=1)
    conf, pred = torch.max(probs, dim=1)

print(f"{classes[pred.item()]}: {conf.item():.2%}")

show_gradcam(
    model=model,
    image_path=image_path,
    device=device,
    class_names=classes,
    target_class_idx=int(pred.item()),
    title=f"Grad-CAM: {classes[pred.item()]} ({conf.item():.1%})",
)


In [ ]:
image_path = r"C:\Users\loaim\OneDrive\Desktop\WhatsApp Image 2026-02-13 at 6.10.31 PM.jpeg"
img = Image.open(image_path).convert("RGB")
img_tensor = get_val_transform()(img)

with torch.no_grad():
    logits = model(img_tensor.unsqueeze(0).to(device))
    probs = F.softmax(logits, dim=1)
    conf, pred = torch.max(probs, dim=1)

print(f"{classes[pred.item()]}: {conf.item():.2%}")

show_gradcam(
    model=model,
    image_path=image_path,
    device=device,
    class_names=classes,
    target_class_idx=int(pred.item()),
    title=f"Grad-CAM: {classes[pred.item()]} ({conf.item():.1%})",
)


In [ ]:
image_path = r"C:\Users\loaim\OneDrive\Desktop\img2.jpeg"
img = Image.open(image_path).convert("RGB")
img_tensor = get_val_transform()(img)

with torch.no_grad():
    logits = model(img_tensor.unsqueeze(0).to(device))
    probs = F.softmax(logits, dim=1)
    conf, pred = torch.max(probs, dim=1)

print(f"{classes[pred.item()]}: {conf.item():.2%}")

show_gradcam(
    model=model,
    image_path=image_path,
    device=device,
    class_names=classes,
    target_class_idx=int(pred.item()),
    title=f"Grad-CAM: {classes[pred.item()]} ({conf.item():.1%})",
)


---

## 16. Inference + Calibration 

1. Restore model and class mapping from checkpoint artifacts.
2. Fit temperature scaling on validation logits.
3. Run uncertainty-aware inference with TTA and class-specific thresholds.
4. Tune thresholds on a real-world holdout.

In [ ]:


# Resolve device
if 'device' not in globals():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Pick checkpoint path (prefer fine-tuned path if it exists in notebook globals)
candidate_ckpts = []
if 'MOBILENET_FINETUNED_SAVE_PATH' in globals():
    candidate_ckpts.append(MOBILENET_FINETUNED_SAVE_PATH)
candidate_ckpts.extend([
   "../models/MobileNet_Finetuned_added_data.pth",
])

checkpoint_path = None
for c in candidate_ckpts:
    if c and os.path.exists(c):
        checkpoint_path = c
        break

if checkpoint_path is None:
    raise FileNotFoundError('Could not find a MobileNet checkpoint path')

ckpt = torch.load(checkpoint_path, map_location=device)

# Resolve class mapping from checkpoint -> metadata -> existing classes
if isinstance(ckpt, dict) and 'idx_to_class' in ckpt:
    idx_to_class_runtime = list(ckpt['idx_to_class'])
elif isinstance(ckpt, dict) and 'class_to_idx' in ckpt:
    inv = {v: k for k, v in ckpt['class_to_idx'].items()}
    idx_to_class_runtime = [inv[i] for i in range(len(inv))]
elif os.path.exists('../models/model_metadata.json'):
    with open('../models/model_metadata.json', 'r', encoding='utf-8') as f:
        idx_to_class_runtime = list(json.load(f).get('classes', []))
elif 'classes' in globals():
    idx_to_class_runtime = list(classes)
else:
    raise RuntimeError('Could not resolve class names from artifacts or runtime')

classes = idx_to_class_runtime
num_classes = len(classes)

# Rebuild/load model
model = get_mobilenet_model(num_classes, version='v2', dropout=0.2).to(device)
if isinstance(ckpt, dict) and 'model_state' in ckpt:
    model.load_state_dict(ckpt['model_state'])
else:
    model.load_state_dict(ckpt)
model.eval()

print(f'Loaded model from: {checkpoint_path}')
print(f'Classes: {num_classes}')


In [ ]:
def collect_logits_and_labels(model, dataloader, device):
    model.eval()
    logits_list = []
    labels_list = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            logits_list.append(logits)
            labels_list.append(labels)

    return torch.cat(logits_list, dim=0), torch.cat(labels_list, dim=0)


def expected_calibration_error(probs, labels, n_bins=15):
    confidences, predictions = probs.max(dim=1)
    accuracies = predictions.eq(labels)
    bin_edges = torch.linspace(0, 1, n_bins + 1, device=probs.device)
    ece = torch.zeros(1, device=probs.device)

    for i in range(n_bins):
        lower = bin_edges[i]
        upper = bin_edges[i + 1]
        in_bin = (confidences > lower) & (confidences <= upper)
        prop_in_bin = in_bin.float().mean()
        if prop_in_bin.item() > 0:
            acc_in_bin = accuracies[in_bin].float().mean()
            conf_in_bin = confidences[in_bin].mean()
            ece += torch.abs(conf_in_bin - acc_in_bin) * prop_in_bin

    return float(ece.item())


def fit_temperature(logits, labels, max_iter=100):
    temperature = torch.ones(1, device=logits.device, requires_grad=True)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.LBFGS([temperature], lr=0.01, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        t = torch.clamp(temperature, 0.5, 5.0)
        loss = criterion(logits / t, labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(torch.clamp(temperature.detach(), 0.5, 5.0).item())


logits_val, labels_val = collect_logits_and_labels(model, val_loader, device)
probs_before = F.softmax(logits_val, dim=1)
temperature_opt = fit_temperature(logits_val, labels_val)
probs_after = F.softmax(logits_val / temperature_opt, dim=1)

print(f'Optimal temperature: {temperature_opt:.4f}')
print(f'NLL before: {F.cross_entropy(logits_val, labels_val).item():.4f} | after: {F.cross_entropy(logits_val / temperature_opt, labels_val).item():.4f}')
print(f'ECE before: {expected_calibration_error(probs_before, labels_val):.4f} | after: {expected_calibration_error(probs_after, labels_val):.4f}')


## 17. Uncertainty Inference (TTA + Class Thresholds)


In [ ]:
DEFAULT_CONF_THRESHOLD = 0.40
DEFAULT_MARGIN_THRESHOLD = 0.10
FAMILY_GAP_THRESHOLD = 0.30

CLASS_THRESHOLDS = {
    'citrus_black_spot': {'conf': 0.55, 'margin': 0.18},
    'citrus_foliage_damage': {'conf': 0.50, 'margin': 0.15},
    'citrus_mealybugs': {'conf': 0.50, 'margin': 0.15},
    'citrus_greening': {'conf': 0.50, 'margin': 0.15},
    'mango_healthy': {'conf': 0.35, 'margin': 0.08},
    'mango_sooty_mould': {'conf': 0.50, 'margin': 0.15},
}


def _tta_variants(pil_img):
    return [
        pil_img,
        pil_img.transpose(Image.FLIP_LEFT_RIGHT),
        pil_img.rotate(8),
        pil_img.rotate(-8),
    ]


def get_thresholds_for_prediction(predicted_class):
    th = CLASS_THRESHOLDS.get(predicted_class)
    if th is None:
        return DEFAULT_CONF_THRESHOLD, DEFAULT_MARGIN_THRESHOLD
    return th['conf'], th['margin']


def predict_with_uncertainty_tta(
    image_path,
    model,
    idx_to_class,
    device,
    temperature=1.0,
    class_thresholds=None,
    use_tta=True,
):
    img = Image.open(image_path).convert('RGB')
    transform = get_val_transform()

    with torch.no_grad():
        if use_tta:
            probs_list = []
            for v in _tta_variants(img):
                x = transform(v).unsqueeze(0).to(device)
                logits = model(x)
                probs_list.append(F.softmax(logits / temperature, dim=1))
            probs = torch.mean(torch.cat(probs_list, dim=0), dim=0)
        else:
            x = transform(img).unsqueeze(0).to(device)
            logits = model(x)
            probs = F.softmax(logits / temperature, dim=1).squeeze(0)

    top2_probs, top2_idx = torch.topk(probs, k=2)
    top1_conf = float(top2_probs[0].item())
    top2_conf = float(top2_probs[1].item())
    margin = top1_conf - top2_conf

    pred_idx = int(top2_idx[0].item())
    pred_class = idx_to_class[pred_idx]

    conf_th = DEFAULT_CONF_THRESHOLD
    margin_th = DEFAULT_MARGIN_THRESHOLD
    if class_thresholds is not None:
        conf_th, margin_th = get_thresholds_for_prediction(pred_class)

    uncertain = (top1_conf < conf_th) or (margin < margin_th)

    # Family-consistency safeguard
    citrus_prob = float(sum(probs[i].item() for i, n in enumerate(idx_to_class) if n.startswith('citrus_')))
    mango_prob = float(sum(probs[i].item() for i, n in enumerate(idx_to_class) if n.startswith('mango_')))
    family_gap = abs(citrus_prob - mango_prob)
    family_rule_triggered = False
    if family_gap < FAMILY_GAP_THRESHOLD:
        uncertain = True
        family_rule_triggered = True

    pair_rule_triggered = False
    pair_required_margin = None
    pair_key = frozenset((pred_class, idx_to_class[int(top2_idx[1].item())]))
    if 'PAIR_MARGIN_RULES' in globals() and pair_key in PAIR_MARGIN_RULES:
        pair_required_margin = float(PAIR_MARGIN_RULES[pair_key])
        if margin < pair_required_margin:
            uncertain = True
            pair_rule_triggered = True

    return {
        'predicted_class': pred_class,
        'predicted_index': pred_idx,
        'confidence': top1_conf,
        'second_best_class': idx_to_class[int(top2_idx[1].item())],
        'second_best_confidence': top2_conf,
        'margin': margin,
        'conf_threshold_used': conf_th,
        'margin_threshold_used': margin_th,
        'uncertain': bool(uncertain),
        'decision': 'uncertain' if uncertain else 'accept',
        'tta_used': bool(use_tta),
        'pair_rule_triggered': bool(pair_rule_triggered),
        'pair_required_margin': pair_required_margin,
        'family_rule_triggered': bool(family_rule_triggered),
        'citrus_probability': citrus_prob,
        'mango_probability': mango_prob,
        'family_gap': family_gap,
        'family_gap_threshold': FAMILY_GAP_THRESHOLD,
    }


print('TTA uncertainty predictor ready')

# Stronger margin rule for known confusion pairs
PAIR_MARGIN_RULES = {
    frozenset(('mango_powdery_mildew', 'mango_sooty_mould')): 0.30,
}


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix


def evaluate_real_world_holdout(
    holdout_root,
    model,
    idx_to_class,
    device,
    temperature=1.0,
    class_thresholds=None,
    use_tta=True,
):
    class_to_idx = {c: i for i, c in enumerate(idx_to_class)}
    image_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

    y_true = []
    y_pred = []
    rejected = 0

    for class_name in sorted(os.listdir(holdout_root)):
        class_dir = os.path.join(holdout_root, class_name)
        if not os.path.isdir(class_dir) or class_name not in class_to_idx:
            continue

        for fn in os.listdir(class_dir):
            if not fn.lower().endswith(image_exts):
                continue
            out = predict_with_uncertainty_tta(
                image_path=os.path.join(class_dir, fn),
                model=model,
                idx_to_class=idx_to_class,
                device=device,
                temperature=temperature,
                class_thresholds=class_thresholds,
                use_tta=use_tta,
            )
            y_true.append(class_to_idx[class_name])
            if out['uncertain']:
                rejected += 1
                y_pred.append(-1)
            else:
                y_pred.append(out['predicted_index'])

    total = len(y_true)
    if total == 0:
        raise ValueError(f'No images found under: {holdout_root}')

    accepted_mask = [p != -1 for p in y_pred]
    y_true_acc = [t for t, keep in zip(y_true, accepted_mask) if keep]
    y_pred_acc = [p for p in y_pred if p != -1]

    print(f'Total samples: {total}')
    print(f'Rejected: {rejected} ({rejected/total:.2%})')
    print(f'Accepted: {len(y_pred_acc)} ({len(y_pred_acc)/total:.2%})')

    if y_pred_acc:
        print('Classification report on accepted predictions:')
        print(classification_report(
            y_true_acc,
            y_pred_acc,
            labels=list(range(len(idx_to_class))),
            target_names=idx_to_class,
            zero_division=0,
        ))
        print('Confusion matrix on accepted predictions:')
        print(confusion_matrix(y_true_acc, y_pred_acc, labels=list(range(len(idx_to_class)))))
    else:
        print('No accepted predictions; thresholds are too strict.')


def grid_search_thresholds(
    holdout_root,
    model,
    idx_to_class,
    device,
    temperature=1.0,
    conf_grid=(0.30, 0.35, 0.40, 0.45, 0.50),
    margin_grid=(0.05, 0.08, 0.10, 0.12, 0.15),
    use_tta=True,
):
    rows = []
    global DEFAULT_CONF_THRESHOLD, DEFAULT_MARGIN_THRESHOLD

    original_conf = DEFAULT_CONF_THRESHOLD
    original_margin = DEFAULT_MARGIN_THRESHOLD

    for c in conf_grid:
        for m in margin_grid:
            DEFAULT_CONF_THRESHOLD = float(c)
            DEFAULT_MARGIN_THRESHOLD = float(m)

            class_to_idx = {x: i for i, x in enumerate(idx_to_class)}
            image_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
            total = 0
            accepted = 0
            correct_accepted = 0

            for class_name in sorted(os.listdir(holdout_root)):
                class_dir = os.path.join(holdout_root, class_name)
                if not os.path.isdir(class_dir) or class_name not in class_to_idx:
                    continue
                for fn in os.listdir(class_dir):
                    if not fn.lower().endswith(image_exts):
                        continue
                    total += 1
                    out = predict_with_uncertainty_tta(
                        image_path=os.path.join(class_dir, fn),
                        model=model,
                        idx_to_class=idx_to_class,
                        device=device,
                        temperature=temperature,
                        class_thresholds=None,
                        use_tta=use_tta,
                    )
                    if not out['uncertain']:
                        accepted += 1
                        if out['predicted_class'] == class_name:
                            correct_accepted += 1

            accepted_rate = accepted / total if total else 0.0
            accepted_acc = (correct_accepted / accepted) if accepted else 0.0
            rows.append({
                'conf_threshold': float(c),
                'margin_threshold': float(m),
                'accepted_rate': accepted_rate,
                'reject_rate': 1.0 - accepted_rate,
                'accepted_accuracy': accepted_acc,
                'coverage_x_accuracy': accepted_rate * accepted_acc,
            })

    DEFAULT_CONF_THRESHOLD = original_conf
    DEFAULT_MARGIN_THRESHOLD = original_margin

    import pandas as pd
    return pd.DataFrame(rows).sort_values(['accepted_accuracy', 'coverage_x_accuracy'], ascending=False)


print('Holdout evaluator + threshold grid search ready')


In [ ]:
from PIL import Image

# Example inference on a real image
sample_path = r"C:\Users\loaim\OneDrive\Desktop\mango_powdery_mildew.jpg"
out = predict_with_uncertainty_tta(
    image_path=sample_path,
    model=model,
    idx_to_class=idx_to_class_runtime,
    device=device,
    temperature=temperature_opt,
    class_thresholds=CLASS_THRESHOLDS,
    use_tta=True,
)
print(out)


In [ ]:
# 1) verify model-class mapping is from checkpoint
ckpt = torch.load("../models/MobileNet_Finetuned_added_data.pth", map_location="cpu")
print("idx_to_class from ckpt:", ckpt.get("idx_to_class", None))
print("class_to_idx from ckpt:", ckpt.get("class_to_idx", None))

# 2) compare your runtime mapping
print("runtime classes:", idx_to_class_runtime)

# 3) print top-5 for the problematic image
img_path = r"C:\Users\loaim\OneDrive\Desktop\citruscanker.jpg"
out = predict_with_uncertainty_tta(
    image_path=img_path,
    model=model,
    idx_to_class=idx_to_class_runtime,
    device=device,
    temperature=temperature_opt,
    class_thresholds=CLASS_THRESHOLDS,
    use_tta=True,
)
print(out)


## 19. Targeted Fine-Tuning For Real-World Inference

This section is designed for current situation: strong validation metrics but real-world mistakes.

What it does:
1. Optionally loads additional hard examples from `data/hard/train` and `data/hard/val`.
2. Fine-tunes from your current best checkpoint.
3. Uses low learning rates and only updates classifier + last two feature blocks.
4. Adds slight extra loss emphasis for known confusing citrus classes (`0,2,3,5`).

Why this helps inference:
- It adapts the model to edge-case samples without heavily changing the already good global validation behavior.


In [ ]:
import os
import copy
import torch
import numpy as np
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, ConcatDataset, Dataset
from torchvision.datasets import ImageFolder


HARD_TRAIN_DIR = '../data/hard/train'
HARD_VAL_DIR = '../data/hard/val'

FINETUNE_EPOCHS = 4
FINETUNE_LR_HEAD = 1e-4
FINETUNE_LR_BACKBONE = 1e-5
FINETUNE_BATCH_SIZE = 64

# Only fine-tune on the confusion pairs you are actively trying to fix.
CONFUSION_PAIRS = [
    ('mango_powdery_mildew', 'mango_sooty_mould'),
]
ACTIVE_HARD_CLASSES = sorted({cls for pair in CONFUSION_PAIRS for cls in pair})

print('Targeted fine-tuning config:')
print(f'  epochs={FINETUNE_EPOCHS}, lr_head={FINETUNE_LR_HEAD}, lr_backbone={FINETUNE_LR_BACKBONE}')
print(f'  hard_train_dir exists: {os.path.exists(HARD_TRAIN_DIR)}')
print(f'  hard_val_dir exists  : {os.path.exists(HARD_VAL_DIR)}')
print('  active hard classes  :', ACTIVE_HARD_CLASSES)


In [ ]:
def _load_optional_hard_dataset(path, transform, expected_class_to_idx, active_classes=None):
    path = Path(path)
    if not path.exists():
        return None

    # Load without requiring every base class to exist in the hard-set root.
    ds = ImageFolder(root=str(path), transform=transform)
    if len(ds) == 0:
        return None

    allowed = set(expected_class_to_idx)
    present = set(ds.class_to_idx)
    unknown = present - allowed
    if unknown:
        raise ValueError(f'Unknown hard-set classes in {path}: {sorted(unknown)}')

    if active_classes is not None:
        active_classes = set(active_classes)
        missing = active_classes - allowed
        if missing:
            raise ValueError(f'Active hard classes not in base mapping: {sorted(missing)}')
        filtered_samples = [(p, y) for p, y in ds.samples if ds.classes[y] in active_classes]
        if not filtered_samples:
            return None
        ds = RemappedImageFolder(ds, expected_class_to_idx, filtered_samples=filtered_samples)
    elif ds.class_to_idx != expected_class_to_idx:
        ds = RemappedImageFolder(ds, expected_class_to_idx)

    return ds


In [ ]:
# Build standalone targeted fine-tune datasets from current train/val plus optional hard sets
base_train = ImageFolder(root=TRAIN_DIR, transform=get_light_transform())
base_val = ImageFolder(root=VAL_DIR, transform=get_val_transform())

hard_train_ds = _load_optional_hard_dataset(HARD_TRAIN_DIR, get_light_transform(), base_train.class_to_idx, active_classes=ACTIVE_HARD_CLASSES)
hard_val_ds = _load_optional_hard_dataset(HARD_VAL_DIR, get_val_transform(), base_val.class_to_idx, active_classes=ACTIVE_HARD_CLASSES)

train_parts = [base_train]
val_parts = [base_val]
if hard_train_ds is not None:
    train_parts.append(hard_train_ds)
if hard_val_ds is not None:
    val_parts.append(hard_val_ds)

train_ds = ConcatDataset(train_parts)
val_ds = ConcatDataset(val_parts)

print(f'Fine-tune train dataset size: {len(train_ds)}')
print(f'Fine-tune val dataset size  : {len(val_ds)}')


In [ ]:
# Build label list for class weighting from combined train dataset

def _extract_labels_from_concat(ds):
    labels = []
    for sub in ds.datasets:
        if hasattr(sub, 'samples'):
            labels.extend([y for _, y in sub.samples])
        else:
            for _, y in sub:
                labels.append(int(y))
    return labels

combined_train_labels = _extract_labels_from_concat(train_ds)
num_classes = len(classes)

class_counts = np.bincount(combined_train_labels, minlength=num_classes)
weights = 1.0 / (class_counts + 1e-6)

# Emphasize only the active confusion-pair classes for this run.
active_hard_indices = [base_train.class_to_idx[c] for c in ACTIVE_HARD_CLASSES if c in base_train.class_to_idx]
for idx in active_hard_indices:
    if idx < len(weights):
        weights[idx] *= 1.20

weights = torch.tensor(weights, dtype=torch.float32, device=device)
criterion_ft = nn.CrossEntropyLoss(weight=weights)

print('Weighted CE for targeted fine-tuning prepared')
print('Boosted hard-class indices:', active_hard_indices)


In [ ]:
# Load best checkpoint as starting point
ckpt_path_candidates = [
    globals().get('MOBILENET_FINETUNED_SAVE_PATH', None),
    '../models/MobileNet_Finetuned_added_data.pth',
    '../models/MobileNet_Finetuned.pth',
    MOBILENET_MODEL_SAVE_PATH,
]
ckpt_path = next((x for x in ckpt_path_candidates if x and os.path.exists(x)), None)
if ckpt_path is None:
    raise FileNotFoundError('No checkpoint found for targeted fine-tuning')

ckpt = torch.load(ckpt_path, map_location=device)
model_ft = get_mobilenet_model(num_classes, version='v2', dropout=0.2).to(device)

if isinstance(ckpt, dict) and 'model_state' in ckpt:
    model_ft.load_state_dict(ckpt['model_state'])
else:
    model_ft.load_state_dict(ckpt)

# Freeze all, then unfreeze classifier + last two feature blocks
for p in model_ft.parameters():
    p.requires_grad = False
for p in model_ft.classifier.parameters():
    p.requires_grad = True
for p in model_ft.features[-2:].parameters():
    p.requires_grad = True

optimizer_ft = torch.optim.Adam([
    {'params': model_ft.classifier.parameters(), 'lr': FINETUNE_LR_HEAD},
    {'params': model_ft.features[-2:].parameters(), 'lr': FINETUNE_LR_BACKBONE},
])

train_loader_ft = DataLoader(train_ds, batch_size=FINETUNE_BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader_ft = DataLoader(val_ds, batch_size=FINETUNE_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Starting targeted fine-tune from: {ckpt_path}')
print(f'Trainable params: {sum(p.numel() for p in model_ft.parameters() if p.requires_grad):,}')


In [ ]:
# Targeted fine-tuning loop
best_state = copy.deepcopy(model_ft.state_dict())
best_val_acc = 0.0

for epoch in range(FINETUNE_EPOCHS):
    model_ft.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for x, y in train_loader_ft:
        x, y = x.to(device), y.to(device)
        optimizer_ft.zero_grad()
        logits = model_ft(x)
        loss = criterion_ft(logits, y)
        loss.backward()
        optimizer_ft.step()

        train_loss += loss.item() * x.size(0)
        train_correct += (torch.argmax(logits, dim=1) == y).sum().item()
        train_total += y.size(0)

    model_ft.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for x, y in val_loader_ft:
            x, y = x.to(device), y.to(device)
            logits = model_ft(x)
            loss = criterion_ft(logits, y)
            val_loss += loss.item() * x.size(0)
            val_correct += (torch.argmax(logits, dim=1) == y).sum().item()
            val_total += y.size(0)

    tr_loss = train_loss / max(1, train_total)
    tr_acc = train_correct / max(1, train_total)
    va_loss = val_loss / max(1, val_total)
    va_acc = val_correct / max(1, val_total)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = copy.deepcopy(model_ft.state_dict())

    print(f'Epoch {epoch+1}/{FINETUNE_EPOCHS} | '
          f'Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.4f} | '
          f'Val Loss: {va_loss:.4f} | Val Acc: {va_acc:.4f}')

model_ft.load_state_dict(best_state)
print(f'Best fine-tune val acc: {best_val_acc:.4f}')


In [ ]:
# Save targeted fine-tuned checkpoint
TARGETED_FT_SAVE_PATH = '../models/MobileNet_targeted_hard_finetune.pth'

torch.save({
    'model_state': model_ft.state_dict(),
    'idx_to_class': classes,
    'source_checkpoint': ckpt_path,
    'notes': 'targeted hard-example fine-tune (classifier + features[-2:])'
}, TARGETED_FT_SAVE_PATH)

print(f'Saved targeted fine-tuned model to: {TARGETED_FT_SAVE_PATH}')


### Recommended usage

1. Run this section after your normal training/evaluation.
2. Put difficult real-world failures into `data/hard/train/<class_name>/...`.
3. Keep fine-tune short (2-4 epochs). If global val degrades, revert to previous best checkpoint.


## 20. Baseline vs Targeted Fine-Tune Evaluation

This section compares two checkpoints on the same evaluation set:
- Baseline model (before targeted hard-example fine-tuning)
- Targeted fine-tuned model (`MobileNet_targeted_hard_finetune.pth`)

It reports:
1. Overall accuracy comparison
2. Full classification reports
3. Confusion matrices
4. Focused metrics for known difficult citrus classes (`0,2,3,5`)


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.utils.data import DataLoader, ConcatDataset
from torchvision.datasets import ImageFolder

# Candidate baseline checkpoint paths (pick first existing)
baseline_candidates = [
    globals().get('MOBILENET_FINETUNED_SAVE_PATH', None),
    '../models/MobileNet_Finetuned_added_data.pth',
    '../models/MobileNet_Finetuned.pth',
    MOBILENET_MODEL_SAVE_PATH,
]
BASELINE_CKPT = next((x for x in baseline_candidates if x and os.path.exists(x)), None)
TARGETED_CKPT = '../models/MobileNet_targeted_hard_finetune.pth'

if BASELINE_CKPT is None:
    raise FileNotFoundError('No baseline checkpoint found')
if not os.path.exists(TARGETED_CKPT):
    raise FileNotFoundError(f'Targeted checkpoint not found: {TARGETED_CKPT}')

print('Baseline checkpoint:', BASELINE_CKPT)
print('Targeted checkpoint:', TARGETED_CKPT)


In [ ]:
# Build evaluation dataset: base val + optional hard val
base_val_eval = ImageFolder(root=VAL_DIR, transform=get_val_transform())

if 'RemappedImageFolder' not in globals():
    class RemappedImageFolder(Dataset):
        def __init__(self, imagefolder, expected_class_to_idx):
            self.imagefolder = imagefolder
            self.transform = imagefolder.transform
            self.loader = imagefolder.loader
            self.classes = list(expected_class_to_idx.keys())
            self.class_to_idx = dict(expected_class_to_idx)
            self.samples = []
            for path, label in imagefolder.samples:
                class_name = imagefolder.classes[label]
                if class_name not in expected_class_to_idx:
                    raise ValueError(f'Unknown hard-val class {class_name}')
                self.samples.append((path, expected_class_to_idx[class_name]))

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            path, label = self.samples[idx]
            img = self.loader(path)
            if self.transform is not None:
                img = self.transform(img)
            return img, label

hard_val_path = '../data/hard/val'
if os.path.exists(hard_val_path):
    hard_val_raw = ImageFolder(root=hard_val_path, transform=get_val_transform())
    hard_val_eval = RemappedImageFolder(hard_val_raw, base_val_eval.class_to_idx)
    eval_ds = ConcatDataset([base_val_eval, hard_val_eval])
    print(f'Using combined eval set: base({len(base_val_eval)}) + hard({len(hard_val_eval)}) = {len(eval_ds)}')
else:
    eval_ds = base_val_eval
    print(f'Using base eval set only: {len(eval_ds)}')

eval_loader = DataLoader(eval_ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)


In [ ]:
def load_ckpt_model(ckpt_path, num_classes, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    m = get_mobilenet_model(num_classes, version='v2', dropout=0.2).to(device)
    if isinstance(ckpt, dict) and 'model_state' in ckpt:
        m.load_state_dict(ckpt['model_state'])
    else:
        m.load_state_dict(ckpt)
    m.eval()
    return m


def run_eval(model, loader, device):
    y_true, y_pred = [], []
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            y_pred.extend(pred.tolist())
            y_true.extend(y.numpy().tolist())
    return np.array(y_true), np.array(y_pred)


baseline_model_eval = load_ckpt_model(BASELINE_CKPT, len(classes), device)
targeted_model_eval = load_ckpt_model(TARGETED_CKPT, len(classes), device)

y_true_b, y_pred_b = run_eval(baseline_model_eval, eval_loader, device)
y_true_t, y_pred_t = run_eval(targeted_model_eval, eval_loader, device)

assert np.array_equal(y_true_b, y_true_t), 'Evaluation targets mismatch between runs'
y_true = y_true_b

acc_b = accuracy_score(y_true, y_pred_b)
acc_t = accuracy_score(y_true, y_pred_t)

print(f'Baseline accuracy: {acc_b:.4f}')
print(f'Targeted accuracy: {acc_t:.4f}')
print(f'Delta: {acc_t - acc_b:+.4f}')


In [ ]:
print('\n=== Baseline Classification Report ===')
print(classification_report(y_true, y_pred_b, target_names=classes, zero_division=0))

print('=== Targeted Classification Report ===')
print(classification_report(y_true, y_pred_t, target_names=classes, zero_division=0))

cm_b = confusion_matrix(y_true, y_pred_b, labels=list(range(len(classes))))
cm_t = confusion_matrix(y_true, y_pred_t, labels=list(range(len(classes))))

print('=== Baseline Confusion Matrix ===')
print(cm_b)
print('=== Targeted Confusion Matrix ===')
print(cm_t)


In [ ]:
# Focused comparison for difficult citrus classes
focus_idxs = [0, 2, 3, 5]
report_b = classification_report(y_true, y_pred_b, output_dict=True, zero_division=0)
report_t = classification_report(y_true, y_pred_t, output_dict=True, zero_division=0)

rows = []
for idx in focus_idxs:
    key = str(idx)
    rows.append({
        'class_idx': idx,
        'class_name': classes[idx],
        'baseline_precision': report_b[key]['precision'],
        'baseline_recall': report_b[key]['recall'],
        'baseline_f1': report_b[key]['f1-score'],
        'targeted_precision': report_t[key]['precision'],
        'targeted_recall': report_t[key]['recall'],
        'targeted_f1': report_t[key]['f1-score'],
        'delta_f1': report_t[key]['f1-score'] - report_b[key]['f1-score'],
    })

focus_df = pd.DataFrame(rows).sort_values('delta_f1', ascending=False)
print(focus_df.to_string(index=False))


## 21. Productionize Targeted Model

Applies the final recommendations:
1. Promote targeted fine-tuned checkpoint to production alias.
2. Refit temperature on current validation loader for this checkpoint.
3. Save calibration metadata.
4. Run quick checks on known real-world images.
5. Export TorchScript production artifact for API deployment.


In [ ]:
import os
import json
import shutil

PROD_SOURCE_CKPT = '../models/MobileNet_targeted_hard_finetune.pth'
PROD_ALIAS_CKPT = '../models/MobileNet_production.pth'
PROD_TORCHSCRIPT = '../models/mobilenet_v2_production.pt'
PROD_CALIB_PATH = '../models/production_calibration.json'

if not os.path.exists(PROD_SOURCE_CKPT):
    raise FileNotFoundError(f'Targeted checkpoint not found: {PROD_SOURCE_CKPT}')

shutil.copy2(PROD_SOURCE_CKPT, PROD_ALIAS_CKPT)
print(f'Promoted checkpoint to production alias: {PROD_ALIAS_CKPT}')


In [ ]:
# Ensure calibration helpers exist in current kernel state
if 'collect_logits_and_labels' not in globals():
    def collect_logits_and_labels(model, dataloader, device):
        model.eval()
        logits_list = []
        labels_list = []
        with torch.no_grad():
            for images, labels in dataloader:
                images = images.to(device)
                labels = labels.to(device)
                logits = model(images)
                logits_list.append(logits)
                labels_list.append(labels)
        return torch.cat(logits_list, dim=0), torch.cat(labels_list, dim=0)

if 'expected_calibration_error' not in globals():
    def expected_calibration_error(probs, labels, n_bins=15):
        confidences, predictions = probs.max(dim=1)
        accuracies = predictions.eq(labels)
        bin_edges = torch.linspace(0, 1, n_bins + 1, device=probs.device)
        ece = torch.zeros(1, device=probs.device)
        for i in range(n_bins):
            lower = bin_edges[i]
            upper = bin_edges[i + 1]
            in_bin = (confidences > lower) & (confidences <= upper)
            prop_in_bin = in_bin.float().mean()
            if prop_in_bin.item() > 0:
                acc_in_bin = accuracies[in_bin].float().mean()
                conf_in_bin = confidences[in_bin].mean()
                ece += torch.abs(conf_in_bin - acc_in_bin) * prop_in_bin
        return float(ece.item())

if 'fit_temperature' not in globals():
    def fit_temperature(logits, labels, max_iter=100):
        temperature = torch.ones(1, device=logits.device, requires_grad=True)
        criterion = torch.nn.CrossEntropyLoss()
        optimizer = torch.optim.LBFGS([temperature], lr=0.01, max_iter=max_iter)
        def closure():
            optimizer.zero_grad()
            t = torch.clamp(temperature, 0.5, 5.0)
            loss = criterion(logits / t, labels)
            loss.backward()
            return loss
        optimizer.step(closure)
        return float(torch.clamp(temperature.detach(), 0.5, 5.0).item())

# Load production alias checkpoint into model and refit temperature
prod_ckpt = torch.load(PROD_ALIAS_CKPT, map_location=device)
prod_classes = prod_ckpt.get('idx_to_class', classes)

prod_model = get_mobilenet_model(len(prod_classes), version='v2', dropout=0.2).to(device)
prod_model.load_state_dict(prod_ckpt['model_state'])
prod_model.eval()

logits_prod, labels_prod = collect_logits_and_labels(prod_model, val_loader, device)
probs_before_prod = F.softmax(logits_prod, dim=1)
prod_temperature = fit_temperature(logits_prod, labels_prod)
probs_after_prod = F.softmax(logits_prod / prod_temperature, dim=1)

prod_nll_before = F.cross_entropy(logits_prod, labels_prod).item()
prod_nll_after = F.cross_entropy(logits_prod / prod_temperature, labels_prod).item()
prod_ece_before = expected_calibration_error(probs_before_prod, labels_prod)
prod_ece_after = expected_calibration_error(probs_after_prod, labels_prod)

print(f'Production temperature: {prod_temperature:.4f}')
print(f'NLL before: {prod_nll_before:.4f} | after: {prod_nll_after:.4f}')
print(f'ECE before: {prod_ece_before:.4f} | after: {prod_ece_after:.4f}')


In [ ]:
# Quick real-world inference checks on known examples (edit paths as needed)
known_real_images = [
    r'C:/Users/loaim/OneDrive/Desktop/img2.jpeg',
    r'C:/Users/loaim/OneDrive/Desktop/mango_powdery_mildew.jpg',
]

for img_path in known_real_images:
    if not os.path.exists(img_path):
        print(f'SKIP (missing): {img_path}')
        continue

    out = predict_with_uncertainty_tta(
        image_path=img_path,
        model=prod_model,
        idx_to_class=prod_classes,
        device=device,
        temperature=prod_temperature,
        class_thresholds=CLASS_THRESHOLDS if 'CLASS_THRESHOLDS' in globals() else None,
        use_tta=True,
    )

    print('=' * 80)
    print('IMAGE:', img_path)
    print(out)


In [ ]:
# Save production calibration metadata
production_meta = {
    'checkpoint': PROD_ALIAS_CKPT,
    'torchscript': PROD_TORCHSCRIPT,
    'temperature': float(prod_temperature),
    'confidence_threshold': float(DEFAULT_CONF_THRESHOLD) if 'DEFAULT_CONF_THRESHOLD' in globals() else 0.4,
    'margin_threshold': float(DEFAULT_MARGIN_THRESHOLD) if 'DEFAULT_MARGIN_THRESHOLD' in globals() else 0.1,
    'nll_before': float(prod_nll_before),
    'nll_after': float(prod_nll_after),
    'ece_before': float(prod_ece_before),
    'ece_after': float(prod_ece_after),
    'classes': list(prod_classes),
}

with open(PROD_CALIB_PATH, 'w', encoding='utf-8') as f:
    json.dump(production_meta, f, indent=2)

print(f'Saved production calibration metadata: {PROD_CALIB_PATH}')


In [ ]:
# Export TorchScript production artifact for deployment API
prod_model.eval()
example = torch.randn(1, 3, 224, 224).to(device)
traced_prod = torch.jit.trace(prod_model, example)
traced_prod.save(PROD_TORCHSCRIPT)
print(f'Exported production TorchScript model: {PROD_TORCHSCRIPT}')


### Deployment note

To use this model in API deployment:
- Point API model loader to `../models/mobilenet_v2_production.pt` (or copied path in deployment/models).
- Use temperature from `../models/production_calibration.json`.


## 22. Leaf Isolation Segmentation Experiment

This section tests whether removing background changes the model's decisions on hard real-world images. It uses a lightweight classical leaf mask (HSV + GrabCut fallback), so we can validate the idea before introducing a heavier segmentation model.


In [ ]:
import os
import io
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Optional hard examples to compare. Update this list to your current failure cases.
SEGMENT_TEST_IMAGES = [
    r"C:/Users/loaim/OneDrive/Desktop/img2.jpeg",
]
SEGMENT_OUTPUT_DIR = Path("../outputs/segmentation_experiment")
SEGMENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from rembg import new_session, remove
    REMBG_AVAILABLE = True
except Exception:
    REMBG_AVAILABLE = False

_REMBG_SESSION = None


def _get_rembg_session():
    global _REMBG_SESSION
    if not REMBG_AVAILABLE:
        return None
    if _REMBG_SESSION is None:
        _REMBG_SESSION = new_session("u2net")
    return _REMBG_SESSION


def _largest_component(mask: np.ndarray) -> np.ndarray:
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    if num_labels <= 1:
        return mask
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == largest).astype(np.uint8)


def build_leaf_mask_grabcut(image_bgr: np.ndarray) -> np.ndarray:
    """Fallback classical mask using rectangle-initialized GrabCut."""
    h, w = image_bgr.shape[:2]
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    lower = np.array([20, 20, 20], dtype=np.uint8)
    upper = np.array([100, 255, 255], dtype=np.uint8)
    hsv_mask = cv2.inRange(hsv, lower, upper)

    kernel = np.ones((5, 5), np.uint8)
    hsv_mask = cv2.morphologyEx(hsv_mask, cv2.MORPH_OPEN, kernel)
    hsv_mask = cv2.morphologyEx(hsv_mask, cv2.MORPH_CLOSE, kernel)

    grab_mask = np.full((h, w), cv2.GC_PR_BGD, dtype=np.uint8)
    pad_x = max(10, int(0.08 * w))
    pad_y = max(10, int(0.08 * h))
    rect = (pad_x, pad_y, max(1, w - 2 * pad_x), max(1, h - 2 * pad_y))
    grab_mask[pad_y:pad_y + rect[3], pad_x:pad_x + rect[2]] = cv2.GC_PR_FGD
    if np.count_nonzero(hsv_mask) > 0:
        grab_mask[hsv_mask > 0] = cv2.GC_FGD

    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    try:
        cv2.grabCut(image_bgr, grab_mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_MASK)
    except cv2.error:
        fallback = np.zeros((h, w), dtype=np.uint8)
        fallback[pad_y:pad_y + rect[3], pad_x:pad_x + rect[2]] = 1
        return fallback

    mask = np.where(
        (grab_mask == cv2.GC_FGD) | (grab_mask == cv2.GC_PR_FGD),
        1,
        0,
    ).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = _largest_component(mask)

    if mask.sum() == 0:
        fallback = np.zeros((h, w), dtype=np.uint8)
        fallback[pad_y:pad_y + rect[3], pad_x:pad_x + rect[2]] = 1
        return fallback
    return mask


def build_leaf_mask(image_bgr: np.ndarray) -> np.ndarray:
    """Foreground mask using rembg when available, with GrabCut fallback."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    if REMBG_AVAILABLE:
        try:
            session = _get_rembg_session()
            # Ask rembg for the mask directly. This is more reliable than inferring
            # alpha from a returned cutout image.
            rembg_mask = remove(image_rgb, session=session, only_mask=True)

            if isinstance(rembg_mask, bytes):
                rembg_mask = np.array(Image.open(io.BytesIO(rembg_mask)).convert("L"))
            elif isinstance(rembg_mask, Image.Image):
                rembg_mask = np.array(rembg_mask.convert("L"))
            elif isinstance(rembg_mask, np.ndarray):
                if rembg_mask.ndim == 3:
                    rembg_mask = rembg_mask[..., 0]
            else:
                rembg_mask = None

            if rembg_mask is not None:
                mask = (rembg_mask > 10).astype(np.uint8)
                kernel = np.ones((5, 5), np.uint8)
                mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
                mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
                mask = _largest_component(mask)
                if mask.sum() > 0:
                    return mask
                print("rembg returned an empty mask, falling back to GrabCut")
        except Exception as e:
            print(f"rembg failed, falling back to GrabCut: {type(e).__name__}: {e}")

    return build_leaf_mask_grabcut(image_bgr)


def apply_mask_black_background(image_rgb: np.ndarray, mask: np.ndarray) -> np.ndarray:
    masked = image_rgb.copy()
    masked[mask == 0] = 0
    return masked


def visualize_leaf_mask(image_path: str, save_dir: Path = SEGMENT_OUTPUT_DIR):
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mask = build_leaf_mask(image_bgr)
    masked_rgb = apply_mask_black_background(image_rgb, mask)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(image_rgb)
    axes[0].set_title("Original")
    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Leaf Mask")
    axes[2].imshow(masked_rgb)
    axes[2].set_title("Masked Background")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    stem = Path(image_path).stem
    Image.fromarray(masked_rgb).save(save_dir / f"{stem}_masked.png")
    Image.fromarray((mask * 255).astype(np.uint8)).save(save_dir / f"{stem}_mask.png")

    return {
        "image_path": image_path,
        "mask_pixels": int(mask.sum()),
        "masked_image_path": str(save_dir / f"{stem}_masked.png"),
        "mask_path": str(save_dir / f"{stem}_mask.png"),
        "rembg_available": REMBG_AVAILABLE,
    }


In [ ]:
def _segment_predict(image_path: str):
    if "model" not in globals() or "device" not in globals():
        raise RuntimeError("Run the model restore / training cells first so model and device exist.")

    if "idx_to_class_runtime" in globals():
        idx_to_class = idx_to_class_runtime
    elif "classes" in globals():
        idx_to_class = classes
    else:
        raise RuntimeError("No class mapping found. Run the model/class mapping cells first.")

    # Prefer the notebook's uncertainty-aware predictor when available.
    if "predict_with_uncertainty_tta" in globals():
        return predict_with_uncertainty_tta(
            image_path=image_path,
            model=model,
            idx_to_class=idx_to_class,
            device=device,
            temperature=temperature_opt if "temperature_opt" in globals() else 1.0,
            class_thresholds=CLASS_THRESHOLDS if "CLASS_THRESHOLDS" in globals() else None,
            use_tta=True,
        )

    # Fallback: plain top-2 inference so the segmentation experiment is self-contained.
    if "get_val_transform" in globals():
        transform = get_val_transform()
    else:
                transform = get_val_transform()

    import torch
    import torch.nn.functional as F
    from PIL import Image

    image = Image.open(image_path).convert("RGB")
    x = transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        logits = logits / (temperature_opt if "temperature_opt" in globals() else 1.0)
        probs = F.softmax(logits, dim=1).squeeze(0)
        top2_probs, top2_idx = torch.topk(probs, k=min(2, len(idx_to_class)))

    pred_idx = int(top2_idx[0].item())
    second_idx = int(top2_idx[1].item()) if len(top2_idx) > 1 else pred_idx
    pred_conf = float(top2_probs[0].item())
    second_conf = float(top2_probs[1].item()) if len(top2_probs) > 1 else pred_conf

    return {
        "predicted_class": idx_to_class[pred_idx],
        "confidence": pred_conf,
        "second_best_class": idx_to_class[second_idx],
        "second_best_confidence": second_conf,
        "margin": pred_conf - second_conf,
        "decision": "fallback_plain_inference",
        "uncertain": None,
    }


def compare_original_vs_masked(image_path: str):
    viz = visualize_leaf_mask(image_path)
    masked_path = viz["masked_image_path"]

    original_result = _segment_predict(image_path)
    masked_result = _segment_predict(masked_path)

    print("Original prediction:")
    print(original_result)
    print("Masked prediction:")
    print(masked_result)

    return {
        "visualization": viz,
        "original": original_result,
        "masked": masked_result,
    }


def batch_compare_original_vs_masked(image_paths):
    rows = []
    for image_path in image_paths:
        if not os.path.exists(image_path):
            print(f"Skipping missing file: {image_path}")
            continue
        result = compare_original_vs_masked(image_path)
        rows.append({
            "image_path": image_path,
            "orig_top1": result["original"].get("predicted_class"),
            "orig_conf": result["original"].get("confidence"),
            "orig_decision": result["original"].get("decision"),
            "masked_top1": result["masked"].get("predicted_class"),
            "masked_conf": result["masked"].get("confidence"),
            "masked_decision": result["masked"].get("decision"),
            "changed": result["original"].get("predicted_class") != result["masked"].get("predicted_class"),
        })
    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        display(df)
        return df
    except Exception:
        return rows


In [ ]:
# Example usage on your current hard images.
# Replace SEGMENT_TEST_IMAGES with any real-world failure cases you want to inspect.
segmentation_results = batch_compare_original_vs_masked(SEGMENT_TEST_IMAGES)
segmentation_results


## 23. ROI Crop Experiment

This section compares original-image predictions against tighter central crops. It is a simpler way to test whether reducing background helps, without relying on segmentation quality.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ROI_OUTPUT_DIR = Path("../outputs/roi_experiment")
ROI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fractions of the original image kept after center crop.
ROI_CROP_SCALES = [0.9, 0.75, 0.6]


def center_crop_image(image: Image.Image, crop_scale: float) -> Image.Image:
    if not (0 < crop_scale <= 1.0):
        raise ValueError("crop_scale must be in (0, 1].")

    w, h = image.size
    crop_w = max(1, int(w * crop_scale))
    crop_h = max(1, int(h * crop_scale))
    left = max(0, (w - crop_w) // 2)
    top = max(0, (h - crop_h) // 2)
    right = left + crop_w
    bottom = top + crop_h
    return image.crop((left, top, right, bottom))


def save_center_crop(image_path: str, crop_scale: float, save_dir: Path = ROI_OUTPUT_DIR) -> str:
    image = Image.open(image_path).convert("RGB")
    cropped = center_crop_image(image, crop_scale)
    stem = Path(image_path).stem
    out_path = save_dir / f"{stem}_crop_{int(crop_scale * 100)}.png"
    cropped.save(out_path)
    return str(out_path)


def visualize_roi_crops(image_path: str, crop_scales=ROI_CROP_SCALES):
    image = Image.open(image_path).convert("RGB")
    variants = [("original", image)]
    for scale in crop_scales:
        variants.append((f"crop_{int(scale * 100)}", center_crop_image(image, scale)))

    fig, axes = plt.subplots(1, len(variants), figsize=(5 * len(variants), 5))
    if len(variants) == 1:
        axes = [axes]
    for ax, (label, img) in zip(axes, variants):
        ax.imshow(np.array(img))
        ax.set_title(label)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
def compare_original_vs_crops(image_path: str, crop_scales=ROI_CROP_SCALES):
    visualize_roi_crops(image_path, crop_scales=crop_scales)

    rows = []
    variants = [("original", image_path)]
    for scale in crop_scales:
        variants.append((f"crop_{int(scale * 100)}", save_center_crop(image_path, scale)))

    for label, variant_path in variants:
        result = _segment_predict(variant_path)
        row = {
            "variant": label,
            "path": variant_path,
            "predicted_class": result.get("predicted_class"),
            "confidence": result.get("confidence"),
            "second_best_class": result.get("second_best_class"),
            "second_best_confidence": result.get("second_best_confidence"),
            "margin": result.get("margin"),
            "decision": result.get("decision"),
        }
        rows.append(row)
        print(label)
        print(result)
        print()

    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        display(df)
        return df
    except Exception:
        return rows


def batch_compare_original_vs_crops(image_paths, crop_scales=ROI_CROP_SCALES):
    all_rows = []
    for image_path in image_paths:
        if not Path(image_path).exists():
            print(f"Skipping missing file: {image_path}")
            continue
        result = compare_original_vs_crops(image_path, crop_scales=crop_scales)
        if hasattr(result, "to_dict"):
            for row in result.to_dict(orient="records"):
                row["image_path"] = image_path
                all_rows.append(row)
        else:
            for row in result:
                row["image_path"] = image_path
                all_rows.append(row)

    try:
        import pandas as pd
        df = pd.DataFrame(all_rows)
        display(df)
        return df
    except Exception:
        return all_rows


In [ ]:
# Example usage on your current hard images.
roi_results = batch_compare_original_vs_crops(SEGMENT_TEST_IMAGES)
roi_results


## 24. Multi-Crop Inference Experiment

This section ensembles the original image with multiple tighter center crops and averages their probabilities. The goal is to see whether background reduction helps when used as a combined inference policy rather than trusting one single crop.


In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image

MULTICROP_SCALES = [1.0, 0.9, 0.75, 0.6]


def _resolve_runtime_classes():
    if "idx_to_class_runtime" in globals():
        return idx_to_class_runtime
    if "classes" in globals():
        return classes
    raise RuntimeError("No class mapping found. Run the model/class mapping cells first.")


def _resolve_val_transform():
    if "get_val_transform" in globals():
        return globals()["get_val_transform"]()
        return get_val_transform()


def _predict_probs_for_pil(pil_image: Image.Image):
    if "model" not in globals() or "device" not in globals():
        raise RuntimeError("Run the model restore / training cells first so model and device exist.")

    transform = _resolve_val_transform()
    x = transform(pil_image.convert("RGB")).unsqueeze(0).to(device)
    temp = temperature_opt if "temperature_opt" in globals() else 1.0

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits / temp, dim=1)
    return probs.squeeze(0)


def predict_with_multicrop(image_path: str, crop_scales=MULTICROP_SCALES):
    idx_to_class = _resolve_runtime_classes()
    image = Image.open(image_path).convert("RGB")

    variant_probs = []
    variant_rows = []

    for scale in crop_scales:
        if scale == 1.0:
            variant = image
            label = "original"
        else:
            variant = center_crop_image(image, scale)
            label = f"crop_{int(scale * 100)}"

        probs = _predict_probs_for_pil(variant)
        variant_probs.append(probs.unsqueeze(0))

        top2_probs, top2_idx = torch.topk(probs, k=min(2, len(idx_to_class)))
        pred_idx = int(top2_idx[0].item())
        second_idx = int(top2_idx[1].item()) if len(top2_idx) > 1 else pred_idx

        variant_rows.append({
            "variant": label,
            "predicted_class": idx_to_class[pred_idx],
            "confidence": float(top2_probs[0].item()),
            "second_best_class": idx_to_class[second_idx],
            "second_best_confidence": float(top2_probs[1].item()) if len(top2_probs) > 1 else float(top2_probs[0].item()),
            "margin": float(top2_probs[0].item() - (top2_probs[1].item() if len(top2_probs) > 1 else top2_probs[0].item())),
        })

    mean_probs = torch.mean(torch.cat(variant_probs, dim=0), dim=0)
    top2_probs, top2_idx = torch.topk(mean_probs, k=min(2, len(idx_to_class)))
    pred_idx = int(top2_idx[0].item())
    second_idx = int(top2_idx[1].item()) if len(top2_idx) > 1 else pred_idx

    aggregate = {
        "predicted_class": idx_to_class[pred_idx],
        "confidence": float(top2_probs[0].item()),
        "second_best_class": idx_to_class[second_idx],
        "second_best_confidence": float(top2_probs[1].item()) if len(top2_probs) > 1 else float(top2_probs[0].item()),
        "margin": float(top2_probs[0].item() - (top2_probs[1].item() if len(top2_probs) > 1 else top2_probs[0].item())),
        "crop_scales": crop_scales,
    }

    return {
        "aggregate": aggregate,
        "variants": variant_rows,
    }


In [ ]:
def compare_single_vs_multicrop(image_path: str, crop_scales=MULTICROP_SCALES):
    single = _segment_predict(image_path)
    multi = predict_with_multicrop(image_path, crop_scales=crop_scales)

    print("Single-image inference:")
    print(single)
    print("Multi-crop aggregate:")
    print(multi["aggregate"])

    try:
        import pandas as pd
        df = pd.DataFrame(multi["variants"])
        display(df)
    except Exception:
        df = multi["variants"]

    return {
        "single": single,
        "multi": multi,
        "variants_table": df,
    }


def batch_compare_single_vs_multicrop(image_paths, crop_scales=MULTICROP_SCALES):
    rows = []
    for image_path in image_paths:
        if not Path(image_path).exists():
            print(f"Skipping missing file: {image_path}")
            continue
        result = compare_single_vs_multicrop(image_path, crop_scales=crop_scales)
        rows.append({
            "image_path": image_path,
            "single_top1": result["single"].get("predicted_class"),
            "single_conf": result["single"].get("confidence"),
            "multi_top1": result["multi"]["aggregate"].get("predicted_class"),
            "multi_conf": result["multi"]["aggregate"].get("confidence"),
            "multi_second": result["multi"]["aggregate"].get("second_best_class"),
            "multi_second_conf": result["multi"]["aggregate"].get("second_best_confidence"),
            "multi_margin": result["multi"]["aggregate"].get("margin"),
            "changed": result["single"].get("predicted_class") != result["multi"]["aggregate"].get("predicted_class"),
        })

    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        display(df)
        return df
    except Exception:
        return rows


In [ ]:
multicrop_results = batch_compare_single_vs_multicrop(SEGMENT_TEST_IMAGES)
multicrop_results
